# 🏨 Hotel Reservation System 2: The Reservation Agent

The coordinator gets its first specialist: a **reservation assistant** that can query the room database and
book rooms through SQL tools. The coordinator decides when to forward: it names `reservation_assistant` in its
reply once the guest has given a room type and dates.

## Learning Objectives
1. **Safe SQL tools** - Model-written queries on a connection that only reads, and bookings through one
   parameterised `book_room` tool, both tested without a model
2. **A tool loop** - `reservation_assistant ⇄ tools` with `tools_condition` deciding when the loop ends
3. **Forwarding by name** - A conditional edge that reads the coordinator's reply with `hc.mentioned_agent()`
4. **Checking the outcome in code** - Reading the `reservations` table to confirm what the agent actually did

## Where this fits
`01_hotel_environment_and_coordinator.ipynb` built the coordinator with memory. This notebook adds the first specialist.
`03_hotel_compliance_checker.ipynb` adds a compliance checker in front of it.

## Prerequisites
- The agents' prompts, printed in `01_hotel_environment_and_coordinator.ipynb` Part 3 (this notebook imports them from `hotel_common.py`)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and imports the building blocks and `hotel_common`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import sqlite3
from typing import Literal

import pandas as pd
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from datetime import date, timedelta

import hotel_common as hc  # shared hotel building blocks, in this folder (see 01_hotel_environment_and_coordinator.ipynb)
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by the coordinator and the reservation agent
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🗄️ Part 1: The Database and Its Tools

The agent writes its own SQL, so the tools must stay safe whatever SQL the model writes. `hc.make_sql_tools()`
(in `hotel_common.py`) returns four plain `sqlite3` tools, with no database wrapper library in between:
- **`list_tables`** and **`describe_table`** show the schema
- **`run_query`** runs ONE query the model wrote, on a connection that only reads, three times over: the file is
  opened read-only (`file:hotel.db?mode=ro`), `PRAGMA query_only = ON` refuses writes, and an **authorizer**
  (`conn.set_authorizer`) allows only reads. SQLite asks the authorizer about every action while it compiles a
  statement, so a write, `ATTACH DATABASE` (which would open *any* SQLite file on this machine) and
  `PRAGMA query_only = OFF` are refused before they run. Read-only flags alone don't stop `ATTACH`. The authorizer
  also hides one column: `reservations.guest_name` reads as NULL (`SQLITE_IGNORE`), so `SELECT *` still works but
  the model sees rooms and dates, never who booked them. Compliance rule 4 (other guests' information stays
  private) is enforced by SQLite here, not only by a prompt.
- **`book_room(guest_name, room_number, start_date, end_date)`** is the only way to write. Its values are bound
  as parameters, never pasted into SQL, and it refuses unreadable or past dates, check-out before check-in, an
  unknown room and a room that is already booked for any of those nights. The check and the insert run as one
  transaction, and a refusal says *when* the room is taken, never *who* took it.

In [ ]:
# ============================================================================
# DATABASE AND SQL TOOLS
# ============================================================================
db_path = hc.init_hotel_db("hotel.db")
sql_tools = hc.make_sql_tools(db_path)  # list_tables, describe_table, run_query (read-only) and book_room
tool = {t.name: t for t in sql_tools}

print(f"✅ {db_path} with tables {tool['list_tables'].invoke({})}")
for t in sql_tools:
    print(f"  🔧 {t.name}: {t.description.splitlines()[0][:90]}")

### The guards, tested without a model
Every guard can be checked by calling the tools directly, the way the agent will. The cell sends an `INSERT`, an
`ATTACH` and a `PRAGMA` through `run_query`, asks it who booked which room, then sends four bad bookings through
`book_room`: one that overlaps Bob Johnson's Family Suite stay, one with check-out before check-in, one in the past
and one for a room that does not exist. Code checks that every attempt was refused, that every guest name came
back as `None`, and that `reservations` did not change.

In [ ]:
# ============================================================================
# THE GUARDS, TESTED WITHOUT A MODEL
# ============================================================================
def count_reservations() -> int:
    with sqlite3.connect("hotel.db") as conn:
        return conn.execute("SELECT COUNT(*) FROM reservations").fetchone()[0]


before = count_reservations()

# 1. run_query only reads: a write, opening another database file and switching the guard off are all refused
attacks = {
    "a write": "INSERT INTO reservations (guest_name, room_number, start_date, end_date) "
               "VALUES ('Mallory', 101, '2031-01-01', '2031-01-02')",
    "opening another database file": "ATTACH DATABASE ':memory:' AS other",
    "switching the guard off": "PRAGMA query_only = OFF",
}
answers = {name: tool["run_query"].invoke({"sql": sql}) for name, sql in attacks.items()}
for name, answer in answers.items():
    print(f"run_query ({name}) -> {answer}")

# 2. run_query never shows who booked a room: guest_name reads as NULL, while SELECT still works
peek = tool["run_query"].invoke({"sql": "SELECT guest_name, room_number FROM reservations"})
print(f"\nrun_query (who booked which room?) ->\n{peek}")
peek_rows = peek.splitlines()[1:]  # after the header line
names_hidden = bool(peek_rows) and all(row.split(" | ")[0] == "None" for row in peek_rows)

# 3. book_room refuses bad bookings (no LLM involved: the tool is called directly)
_, bob_room, bob_in, bob_out = hc.sample_reservations()[1]  # Bob Johnson: room 105, today+30 to today+33
later, today = date.today() + timedelta(days=90), date.today()
bad_bookings = {
    "overlaps Bob's stay": {"room_number": bob_room, "start_date": bob_in, "end_date": bob_out},
    "check-out before check-in": {"room_number": 101, "start_date": str(later), "end_date": str(later - timedelta(days=2))},
    "check-in in the past": {"room_number": 101, "start_date": str(today - timedelta(days=5)), "end_date": str(today)},
    "no such room": {"room_number": 999, "start_date": str(later), "end_date": str(later + timedelta(days=2))},
}
refusals = {name: tool["book_room"].invoke({"guest_name": "Test Guest", **args}) for name, args in bad_bookings.items()}
print()
for name, answer in refusals.items():
    print(f"book_room ({name}) -> {answer}")

after = count_reservations()
print(f"\nReservations before / after: {before} / {after}")
refused = (all(a.startswith("Error") for a in answers.values())
           and all(r.startswith("Cannot book") for r in refusals.values()) and before == after)
print("✅ run_query refused the INSERT, ATTACH and PRAGMA; book_room refused all four bad bookings; no row changed"
      if refused else "❌ an attempt was not refused")
print(f"{'✅' if names_hidden else '❌'} run_query returned {len(peek_rows)} reservations with every guest name as None")

---
## 🤖 Part 2: Two Agents and a Router

- **`conv_assistant`**: the coordinator, now told that `reservation_assistant` exists
- **`reservation_assistant`**: the SQL agent, with the tools bound
- **`choose_next_node`**: a pure function that reads the coordinator's reply and routes on the agent it names

In [ ]:
# ============================================================================
# AGENT NODES, ROUTER AND GRAPH
# ============================================================================
AGENTS = ["reservation_assistant"]
conversation_prompt = hc.conversation_prompt(AGENTS)
llm_with_sql_tools = llm.bind_tools(sql_tools)


def conv_assistant(state: MessagesState) -> dict:
    return {"messages": [llm.invoke([conversation_prompt] + state["messages"])]}


def sql_assistant(state: MessagesState) -> dict:
    return {"messages": [llm_with_sql_tools.invoke([hc.SQL_PROMPT] + state["messages"])]}


def choose_next_node(state: MessagesState) -> Literal["reservation_assistant", "__end__"]:
    return hc.mentioned_agent(state["messages"][-1].content, AGENTS) or END


builder = StateGraph(MessagesState)
builder.add_node("conv_assistant", conv_assistant)
builder.add_node("reservation_assistant", sql_assistant)
builder.add_node("tools", ToolNode(sql_tools))
builder.add_edge(START, "conv_assistant")
builder.add_conditional_edges("conv_assistant", choose_next_node)
builder.add_conditional_edges("reservation_assistant", tools_condition)  # tool call -> tools, else END
builder.add_edge("tools", "reservation_assistant")

graph = builder.compile(checkpointer=InMemorySaver())
print("✅ Graph compiled. Edges:")
hc.print_edges(graph)

---
## ▶️ Part 3: A Scripted Booking

Three guest messages on one thread: an availability question, then the details for a booking. The Family Suite
(room 105) is already booked from today+30 to today+33, so the first request should find it taken.

In [ ]:
# ============================================================================
# SCRIPTED CONVERSATION
# ============================================================================
clash_in = date.today() + timedelta(days=31)
free_in = date.today() + timedelta(days=60)
thread = "booking-demo"

hc.run_turn(graph, f"Is the Family Suite free from {clash_in} to {clash_in + timedelta(days=2)}?", thread)
hc.run_turn(graph, f"Then I'd like a Double room from {free_in} to {free_in + timedelta(days=3)}.", thread)
hc.run_turn(graph, "My name is Dana Lee. Please book it.", thread)

### Did the booking really happen?
The agent's reply is not proof; the database is. Query it directly.

In [ ]:
# ============================================================================
# CHECK THE DATABASE, NOT THE REPLY
# ============================================================================
with sqlite3.connect("hotel.db") as conn:
    rows = pd.read_sql("SELECT * FROM reservations WHERE guest_name LIKE '%Dana%'", conn)
print(rows.to_string(index=False) if len(rows) else "No reservation for Dana Lee was written.")
print(f"\nBooking written: {'✅' if len(rows) == 1 else '❌'}")

**Reading the output:** a ✅ means the agent booked exactly one row for Dana Lee (through `book_room`). A ❌
means it only *said* it booked (or asked another question). This is why the check reads the table. Also look at
the first turn's tool calls: the agent should have queried `reservations` for overlapping dates before answering.

To chat with the system yourself, run `hc.chat_with_agent(graph, "my-thread")` in a new cell.

---
## 📝 Summary

- **Safe SQL tools**: `run_query` runs the model's SQL on a connection that only reads (read-only file,
  `query_only`, an authorizer) that never shows guest names, and `book_room` is the only write, with bound
  parameters and its own checks. The guards were proved by calling the tools directly, with no model.
- **`tools_condition`** + an edge back from `tools` makes the standard tool loop.
- The coordinator forwards by **naming** an agent; `choose_next_node` stays a pure function over its reply.
- Verify side effects in the **database**, not in the agent's wording.

### Next Steps
- `03_hotel_compliance_checker.ipynb`: a compliance checker that screens requests against the hotel's rules